In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_data import load_split


# ------------------------------------------------------------
# Загружаем split и genome
# ------------------------------------------------------------

split = load_split(PROJECT_ROOT)

sequences, alphabet = load_baseline_sequences(split)


# ------------------------------------------------------------
# Только TRAIN contigs
# ------------------------------------------------------------

train_contigs = (
    split.contig_assignment
    .loc[
        split.contig_assignment["split"] == "train",
        "contig"
    ]
    .astype(str)
    .tolist()
)

total_bp = sum(
    len(sequences[contig])
    for contig in train_contigs
)


print("=" * 80)
print("BPE + SOFTMASK EXPERIMENT")
print("=" * 80)

print("Project:", PROJECT_ROOT)
print("Train contigs:", len(train_contigs))
print("Train bp:", f"{total_bp:,}")

print("\nTrain contigs:")

for contig in train_contigs:
    print(
        f"{contig:<20}"
        f"{len(sequences[contig]):>15,} bp"
    )

BPE + SOFTMASK EXPERIMENT
Project: D:\Projects\EPIC\EPIC
Train contigs: 9
Train bp: 159,860,208

Train contigs:
NC_064035.1              22,031,953 bp
NC_064036.1              20,793,841 bp
NC_064038.1              19,000,648 bp
NC_064039.1              18,096,078 bp
NC_064040.1              17,865,913 bp
NC_064043.1              16,593,782 bp
NC_064045.1              16,388,311 bp
NC_064047.1              15,372,318 bp
NC_064048.1              13,717,364 bp


In [2]:
# ============================================================
# CASE-INDEPENDENT DNA + SOFTMASK
# ============================================================

# Исходные FASTA-коды:
#
# 0 A
# 1 C
# 2 G
# 3 T
# 4 a
# 5 c
# 6 g
# 7 t
# 8 N/n
#
# Для BPE:
# A/a -> 0
# C/c -> 1
# G/g -> 2
# T/t -> 3
# N/n -> 4

BASE_LOOKUP = np.array(
    [0, 1, 2, 3, 0, 1, 2, 3, 4],
    dtype=np.uint8,
)

# Отдельно храним:
# 1 = lowercase / softmasked
# 0 = uppercase / normal
#
# Для кода 8 уже невозможно отличить N от n,
# потому что FASTA loader ранее объединил их в один код.
SOFTMASK_LOOKUP = np.array(
    [0, 0, 0, 0, 1, 1, 1, 1, 0],
    dtype=np.uint8,
)


# ------------------------------------------------------------
# Начальное состояние BPE
# ------------------------------------------------------------

bpe_sequences = {}
softmask_counts = {}

for contig in train_contigs:

    raw = sequences[contig]

    # BPE token IDs.
    # uint16 — потому что потом словарь станет >255 токенов.
    bpe_sequences[contig] = (
        BASE_LOOKUP[raw]
        .astype(np.uint16)
    )

    # Пока каждый token = 1 bp,
    # поэтому число lowercase bp внутри token равно 0 или 1.
    #
    # Позже при merge:
    # lower_count(new_token)
    # =
    # lower_count(left) + lower_count(right)
    softmask_counts[contig] = (
        SOFTMASK_LOOKUP[raw]
        .astype(np.uint16)
    )


# ------------------------------------------------------------
# Проверяем статистику
# ------------------------------------------------------------

total_lower = sum(
    int(mask.sum())
    for mask in softmask_counts.values()
)

total_bp = sum(
    len(seq)
    for seq in bpe_sequences.values()
)

print("=" * 80)
print("CASE COLLAPSE + SOFTMASK")
print("=" * 80)

print(f"Train bp:              {total_bp:,}")
print(f"Lowercase bp:          {total_lower:,}")
print(f"Uppercase/N bp:        {total_bp - total_lower:,}")
print(f"Softmask fraction:     {total_lower / total_bp:.4%}")


# ------------------------------------------------------------
# Наглядно смотрим участок, где есть lowercase
# ------------------------------------------------------------

RAW_SYMBOLS = np.array(
    ["A", "C", "G", "T", "a", "c", "g", "t", "N"]
)

BASE_SYMBOLS = np.array(
    ["A", "C", "G", "T", "N"]
)

example_contig = train_contigs[0]

raw = sequences[example_contig]
mask = softmask_counts[example_contig]
base = bpe_sequences[example_contig]

lower_positions = np.flatnonzero(mask)

if len(lower_positions):

    center = int(lower_positions[0])

    start = max(0, center - 25)
    end = min(len(raw), center + 35)

    original_text = "".join(
        RAW_SYMBOLS[raw[start:end]]
    )

    collapsed_text = "".join(
        BASE_SYMBOLS[base[start:end]]
    )

    mask_text = "".join(
        "^" if x else "."
        for x in mask[start:end]
    )

    print("\nExample:")
    print("original :", original_text)
    print("BPE DNA  :", collapsed_text)
    print("softmask :", mask_text)

else:
    print("\nВ первом train-contig lowercase не найден.")

CASE COLLAPSE + SOFTMASK
Train bp:              159,860,208
Lowercase bp:          49,663,429
Uppercase/N bp:        110,196,779
Softmask fraction:     31.0668%

Example:
original : cctaaccctacaccctaaccctaaccctaacccta
BPE DNA  : CCTAACCCTACACCCTAACCCTAACCCTAACCCTA
softmask : ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


In [3]:
mask = softmask_counts[example_contig]
raw = sequences[example_contig]
base = bpe_sequences[example_contig]

# Ищем место, где softmask меняется 0 -> 1 или 1 -> 0
transitions = np.flatnonzero(mask[:-1] != mask[1:])

print("Количество переходов:", len(transitions))

center = int(transitions[0])

start = max(0, center - 30)
end = min(len(raw), center + 31)

print(
    "original :",
    "".join(RAW_SYMBOLS[raw[start:end]])
)

print(
    "BPE DNA  :",
    "".join(BASE_SYMBOLS[base[start:end]])
)

print(
    "softmask :",
    "".join("^" if x else "." for x in mask[start:end])
)

Количество переходов: 192097
original : ctaaccctaaccctaaccctaaccctaactaTCAGAATCGGTCAAATTCCACCTAATTATC
BPE DNA  : CTAACCCTAACCCTAACCCTAACCCTAACTATCAGAATCGGTCAAATTCCACCTAATTATC
softmask : ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^..............................


In [4]:
# ============================================================
# BPE MERGE + OCCURRENCE-SPECIFIC SOFTMASK
# ============================================================

token_text = {
    0: "A",
    1: "C",
    2: "G",
    3: "T",
    4: "N",
}

token_length = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
}


def count_all_pairs(sequences, vocab_size):
    counts = np.zeros(
        vocab_size * vocab_size,
        dtype=np.int64,
    )

    for seq in sequences.values():

        pair_ids = (
            seq[:-1].astype(np.int64) * vocab_size
            + seq[1:].astype(np.int64)
        )

        counts += np.bincount(
            pair_ids,
            minlength=vocab_size * vocab_size,
        )

    return counts


def merge_pair_with_softmask(
    seq,
    lower_counts,
    left,
    right,
    new_token,
):
    """
    Делает non-overlapping BPE merge.

    Одновременно:
        token[left] + token[right] -> new_token

    и:
        lowercase_count[new]
        =
        lowercase_count[left occurrence]
        +
        lowercase_count[right occurrence]
    """

    match = (
        (seq[:-1] == left)
        &
        (seq[1:] == right)
    )

    starts = np.flatnonzero(match)

    if len(starts) == 0:
        return seq, lower_counts, 0

    # --------------------------------------------------------
    # Для X + X возможны перекрытия:
    #
    # A A A A
    # pairs start at 0, 1, 2
    #
    # BPE должен взять:
    # (0,1) и (2,3)
    # --------------------------------------------------------

    if left == right:

        new_run = np.r_[
            True,
            np.diff(starts) != 1
        ]

        run_starts = np.flatnonzero(new_run)

        offsets = (
            np.arange(len(starts))
            -
            np.repeat(
                run_starts,
                np.diff(
                    np.r_[
                        run_starts,
                        len(starts)
                    ]
                )
            )
        )

        starts = starts[offsets % 2 == 0]

    merged_count = len(starts)

    # --------------------------------------------------------
    # TOKEN IDs
    # --------------------------------------------------------

    new_seq = seq.copy()

    new_seq[starts] = new_token

    # --------------------------------------------------------
    # SOFTMASK COUNTS
    # --------------------------------------------------------

    new_lower = lower_counts.copy()

    new_lower[starts] = (
        lower_counts[starts]
        +
        lower_counts[starts + 1]
    )

    # --------------------------------------------------------
    # Удаляем второй элемент каждой merged пары
    # --------------------------------------------------------

    remove = np.zeros(
        len(seq),
        dtype=bool,
    )

    remove[starts + 1] = True

    return (
        new_seq[~remove],
        new_lower[~remove],
        merged_count,
    )


# ============================================================
# ИТЕРАЦИЯ 1 — выбираем пару АВТОМАТИЧЕСКИ
# ============================================================

vocab_size = len(token_text)

counts = count_all_pairs(
    bpe_sequences,
    vocab_size,
)

top_ids = np.argsort(counts)[::-1][:10]


print("=" * 90)
print("BPE ITERATION 1 — TOP-10")
print("=" * 90)

for rank, pair_id in enumerate(top_ids, 1):

    left = int(pair_id // vocab_size)
    right = int(pair_id % vocab_size)

    combined = (
        token_text[left]
        +
        token_text[right]
    )

    print(
        f"{rank:>2}. "
        f"{token_text[left]} + "
        f"{token_text[right]} "
        f"→ {combined:<8} "
        f"| count={counts[pair_id]:>12,}"
    )


# ------------------------------------------------------------
# Самая частая пара
# ------------------------------------------------------------

best_pair = int(top_ids[0])

left = best_pair // vocab_size
right = best_pair % vocab_size

new_token = vocab_size

token_text[new_token] = (
    token_text[left]
    +
    token_text[right]
)

token_length[new_token] = (
    token_length[left]
    +
    token_length[right]
)


print("\nSELECTED:")
print(
    f"{token_text[left]} + {token_text[right]}"
    f" → TOKEN #{new_token}"
    f" = {token_text[new_token]}"
)


# ------------------------------------------------------------
# Merge по всем train contigs
# ------------------------------------------------------------

total_merges = 0

for contig in train_contigs:

    new_seq, new_lower, n = merge_pair_with_softmask(
        bpe_sequences[contig],
        softmask_counts[contig],
        left,
        right,
        new_token,
    )

    bpe_sequences[contig] = new_seq
    softmask_counts[contig] = new_lower

    total_merges += n


print(f"Actual merges: {total_merges:,}")
print(
    f"New token length: "
    f"{token_length[new_token]} bp"
)


# ============================================================
# ПОКАЗЫВАЕМ РЕАЛЬНЫЕ SOFTMASK FRACTIONS
# нового токена
# ============================================================

print("\n" + "=" * 90)
print(
    f"ПРИМЕРЫ TOKEN #{new_token} = "
    f"{token_text[new_token]}"
)
print("=" * 90)

shown = 0

for contig in train_contigs:

    seq = bpe_sequences[contig]
    lower = softmask_counts[contig]

    positions = np.flatnonzero(
        seq == new_token
    )

    for pos in positions[:20]:

        lowercase_count = int(lower[pos])

        fraction = (
            lowercase_count
            /
            token_length[new_token]
        )

        print(
            f"{contig} "
            f"token_pos={pos:>10,} "
            f"| lowercase_count="
            f"{lowercase_count}/{token_length[new_token]} "
            f"| softmask_fraction={fraction:.2f}"
        )

        shown += 1

        if shown >= 20:
            break

    if shown >= 20:
        break

BPE ITERATION 1 — TOP-10
 1. T + T → TT       | count=  15,762,402
 2. A + A → AA       | count=  15,744,606
 3. A + T → AT       | count=  12,947,412
 4. T + A → TA       | count=  11,859,758
 5. C + A → CA       | count=  10,692,952
 6. T + G → TG       | count=  10,660,979
 7. C + T → CT       | count=   9,436,633
 8. A + G → AG       | count=   9,370,438
 9. A + C → AC       | count=   9,303,495
10. G + T → GT       | count=   9,257,832

SELECTED:
T + T → TOKEN #5 = TT
Actual merges: 11,489,404
New token length: 2 bp

ПРИМЕРЫ TOKEN #5 = TT
NC_064035.1 token_pos=     1,675 | lowercase_count=0/2 | softmask_fraction=0.00
NC_064035.1 token_pos=     1,684 | lowercase_count=0/2 | softmask_fraction=0.00
NC_064035.1 token_pos=     1,694 | lowercase_count=0/2 | softmask_fraction=0.00
NC_064035.1 token_pos=     1,701 | lowercase_count=0/2 | softmask_fraction=0.00
NC_064035.1 token_pos=     1,728 | lowercase_count=0/2 | softmask_fraction=0.00
NC_064035.1 token_pos=     1,762 | lowercase_count

In [5]:
# ============================================================
# SOFTMASK DISTRIBUTION ДЛЯ TOKEN #5 = TT
# ============================================================

TOKEN_ID = 5
TOKEN_LEN = token_length[TOKEN_ID]

distribution = np.zeros(TOKEN_LEN + 1, dtype=np.int64)

examples = {
    0: [],
    1: [],
    2: [],
}

for contig in train_contigs:

    seq = bpe_sequences[contig]
    lower = softmask_counts[contig]

    positions = np.flatnonzero(seq == TOKEN_ID)

    values = lower[positions]

    distribution += np.bincount(
        values,
        minlength=TOKEN_LEN + 1
    )

    # Несколько примеров каждого типа
    for pos, lower_count in zip(positions, values):

        lower_count = int(lower_count)

        if len(examples[lower_count]) < 3:
            examples[lower_count].append(
                (contig, int(pos))
            )

        if all(len(v) >= 3 for v in examples.values()):
            break


print("=" * 80)
print("SOFTMASK DISTRIBUTION FOR TT")
print("=" * 80)

total = distribution.sum()

for lower_count, count in enumerate(distribution):

    fraction = lower_count / TOKEN_LEN

    print(
        f"{lower_count}/{TOKEN_LEN}"
        f" | fraction={fraction:.2f}"
        f" | count={count:>12,}"
        f" | share={count / total:>8.3%}"
    )


print("\nEXAMPLES:")

for lower_count, items in examples.items():

    print(
        f"\nsoftmask_fraction="
        f"{lower_count / TOKEN_LEN:.2f}"
    )

    for contig, pos in items:
        print(
            f"  {contig} token_pos={pos:,}"
        )

SOFTMASK DISTRIBUTION FOR TT
0/2 | fraction=0.00 | count=   7,499,668 | share= 65.275%
1/2 | fraction=0.50 | count=      75,743 | share=  0.659%
2/2 | fraction=1.00 | count=   3,913,993 | share= 34.066%

EXAMPLES:

softmask_fraction=0.00
  NC_064035.1 token_pos=1,675
  NC_064035.1 token_pos=1,684
  NC_064035.1 token_pos=1,694

softmask_fraction=0.50
  NC_064035.1 token_pos=3,209
  NC_064035.1 token_pos=4,849
  NC_064035.1 token_pos=6,357

softmask_fraction=1.00
  NC_064035.1 token_pos=2,113
  NC_064035.1 token_pos=2,115
  NC_064035.1 token_pos=2,118


In [6]:
# ============================================================
# AUTOMATIC BPE + SOFTMASK
# продолжаем с iteration 2
# ============================================================

original_bp = sum(
    len(sequences[contig])
    for contig in train_contigs
)

# Первый merge уже сделан
merge_history = [
    (3, 3, 5),  # T + T -> TT
]

TARGET_MERGES = 50
TOP_K = 10


for iteration in range(2, TARGET_MERGES + 1):

    vocab_size = max(token_text) + 1

    # --------------------------------------------------------
    # Считаем пары
    # --------------------------------------------------------

    counts = count_all_pairs(
        bpe_sequences,
        vocab_size,
    )

    top_ids = np.argsort(counts)[::-1][:TOP_K]

    print("\n" + "=" * 100)
    print(f"BPE ITERATION {iteration}")
    print("=" * 100)

    for rank, pair_id in enumerate(top_ids, 1):

        left = int(pair_id // vocab_size)
        right = int(pair_id % vocab_size)

        combined = (
            token_text[left]
            +
            token_text[right]
        )

        combined_length = (
            token_length[left]
            +
            token_length[right]
        )

        print(
            f"{rank:>2}. "
            f"{token_text[left]:<12} + "
            f"{token_text[right]:<12} "
            f"→ {combined:<25} "
            f"| count={counts[pair_id]:>12,} "
            f"| len={combined_length:>3} bp"
        )

    # --------------------------------------------------------
    # Самая частая пара
    # --------------------------------------------------------

    best_pair_id = int(top_ids[0])

    left = best_pair_id // vocab_size
    right = best_pair_id % vocab_size

    new_token = vocab_size

    token_text[new_token] = (
        token_text[left]
        +
        token_text[right]
    )

    token_length[new_token] = (
        token_length[left]
        +
        token_length[right]
    )

    # --------------------------------------------------------
    # Merge + softmask
    # --------------------------------------------------------

    total_merges = 0

    for contig in train_contigs:

        new_seq, new_lower, n = merge_pair_with_softmask(
            bpe_sequences[contig],
            softmask_counts[contig],
            left,
            right,
            new_token,
        )

        bpe_sequences[contig] = new_seq
        softmask_counts[contig] = new_lower

        total_merges += n

    merge_history.append(
        (left, right, new_token)
    )

    # --------------------------------------------------------
    # Общая статистика
    # --------------------------------------------------------

    total_tokens = sum(
        len(seq)
        for seq in bpe_sequences.values()
    )

    compression = original_bp / total_tokens

    # --------------------------------------------------------
    # Softmask статистика НОВОГО токена
    # --------------------------------------------------------

    occurrence_count = 0
    total_lower_bp = 0

    for contig in train_contigs:

        seq = bpe_sequences[contig]
        lower = softmask_counts[contig]

        positions = np.flatnonzero(
            seq == new_token
        )

        occurrence_count += len(positions)

        if len(positions):
            total_lower_bp += int(
                lower[positions].sum()
            )

    mean_softmask_fraction = (
        total_lower_bp
        /
        (
            occurrence_count
            * token_length[new_token]
        )
        if occurrence_count
        else 0.0
    )

    print("\nSELECTED:")

    print(
        f"{token_text[left]} + "
        f"{token_text[right]}"
        f" → TOKEN #{new_token}"
        f" = {token_text[new_token]}"
    )

    print(
        f"Token length:             "
        f"{token_length[new_token]} bp"
    )

    print(
        f"Actual merges:            "
        f"{total_merges:,}"
    )

    print(
        f"Token occurrences now:    "
        f"{occurrence_count:,}"
    )

    print(
        f"Mean softmask fraction:   "
        f"{mean_softmask_fraction:.4f}"
    )

    print(
        f"Tokens remaining:         "
        f"{total_tokens:,}"
    )

    print(
        f"Compression ratio:        "
        f"{compression:.4f}x"
    )

    print(
        f"Average bp/token:         "
        f"{compression:.4f}"
    )


# ============================================================
# SUMMARY
# ============================================================

from collections import Counter

total_tokens = sum(
    len(seq)
    for seq in bpe_sequences.values()
)

compression = original_bp / total_tokens

length_distribution = Counter(
    token_length[token_id]
    for token_id in token_text
    if token_id >= 5
)

print("\n" + "#" * 100)
print("BPE + SOFTMASK SUMMARY")
print("#" * 100)

print(f"Merge rules:            {len(merge_history)}")
print(f"Vocabulary size:        {len(token_text)}")
print(f"Original bp:            {original_bp:,}")
print(f"BPE tokens:             {total_tokens:,}")
print(f"Compression ratio:      {compression:.4f}x")
print(f"Average bp/token:       {compression:.4f}")

print("\nTOKEN LENGTH DISTRIBUTION:")

for length in sorted(length_distribution):

    print(
        f"{length:>3} bp : "
        f"{length_distribution[length]:>4} tokens"
    )

print("\nLONGEST TOKENS:")

longest = sorted(
    (
        token_length[token_id],
        token_id,
        token_text[token_id],
    )
    for token_id in token_text
    if token_id >= 5
)[::-1][:20]

for length, token_id, text in longest:

    print(
        f"#{token_id:<4} "
        f"{text:<35} "
        f"{length:>3} bp"
    )


BPE ITERATION 2
 1. A            + A            → AA                        | count=  15,744,606 | len=  2 bp
 2. C            + A            → CA                        | count=  10,692,952 | len=  2 bp
 3. A            + G            → AG                        | count=   9,370,438 | len=  2 bp
 4. A            + C            → AC                        | count=   9,303,495 | len=  2 bp
 5. T            + A            → TA                        | count=   9,231,294 | len=  2 bp
 6. G            + A            → GA                        | count=   9,068,621 | len=  2 bp
 7. A            + T            → AT                        | count=   8,997,094 | len=  2 bp
 8. T            + G            → TG                        | count=   8,208,389 | len=  2 bp
 9. C            + C            → CC                        | count=   7,437,114 | len=  2 bp
10. G            + G            → GG                        | count=   7,359,235 | len=  2 bp

SELECTED:
A + A → TOKEN #6 = AA
Token leng

In [7]:
# ============================================================
# VERIFY BPE + SOFTMASK INVARIANTS
# ============================================================

reconstructed_bp = 0
reconstructed_lowercase_bp = 0

for contig in train_contigs:

    seq = bpe_sequences[contig]
    lower = softmask_counts[contig]

    # Сколько реальных bp теперь представляют все BPE-токены
    lengths = np.array(
        [token_length[int(token)] for token in seq],
        dtype=np.int64,
    )

    reconstructed_bp += int(lengths.sum())

    # Сколько lowercase bp хранится внутри этих токенов
    reconstructed_lowercase_bp += int(lower.sum())


original_lowercase_bp = 49_663_429


print("=" * 80)
print("BPE INVARIANT CHECK")
print("=" * 80)

print(f"Original bp:               {original_bp:,}")
print(f"Reconstructed bp:          {reconstructed_bp:,}")
print()

print(f"Original lowercase bp:     {original_lowercase_bp:,}")
print(f"Reconstructed lowercase:   {reconstructed_lowercase_bp:,}")
print()

print(
    "DNA preserved:            ",
    reconstructed_bp == original_bp
)

print(
    "Softmask preserved:       ",
    reconstructed_lowercase_bp == original_lowercase_bp
)

print(
    "Global softmask fraction: ",
    f"{reconstructed_lowercase_bp / reconstructed_bp:.4%}"
)

BPE INVARIANT CHECK
Original bp:               159,860,208
Reconstructed bp:          159,860,208

Original lowercase bp:     49,663,429
Reconstructed lowercase:   49,663,429

DNA preserved:             True
Softmask preserved:        True
Global softmask fraction:  31.0668%


In [8]:
# ============================================================
# SAVE TRAINED BPE-50 TOKENIZER
# ============================================================

import torch

BPE_DIR = PROJECT_ROOT / "artifacts"
BPE_DIR.mkdir(parents=True, exist_ok=True)

BPE_PATH = BPE_DIR / "bpe_softmask_50.pt"

bpe_artifact = {
    "merge_history": merge_history,
    "token_text": token_text,
    "token_length": token_length,
    "vocab_size": len(token_text),
    "num_merges": len(merge_history),
}

torch.save(
    bpe_artifact,
    BPE_PATH,
)

print("=" * 80)
print("BPE TOKENIZER SAVED")
print("=" * 80)

print("Path:        ", BPE_PATH)
print("Merge rules: ", len(merge_history))
print("Vocab size:  ", len(token_text))
print("Max token bp:", max(token_length.values()))

BPE TOKENIZER SAVED
Path:         D:\Projects\EPIC\EPIC\artifacts\bpe_softmask_50.pt
Merge rules:  50
Vocab size:   55
Max token bp: 4
